# Exploration 01 — OptionMetrics `vsurfd` (surface de volatilite)

Objectif : localiser la table de surface de vol, lire ses **vraies colonnes**, confirmer que **91 jours / delta 50** existent, puis (etape suivante) trouver le `secid` du SPX.

Read-only — aucune donnee lourde telechargee (DISTINCT sur colonnes a faible cardinalite).

In [1]:
from dispersion.data.wrds_client import get_connection

db = get_connection()

Loading library list...
Done


## 1) Ou vit `vsurfd` ? (decoupage annuel ?)

In [2]:
for lib in ["optionm_all", "optionm"]:
    try:
        tables = db.list_tables(library=lib)
    except Exception as e:
        print(f"[{lib}] inaccessible: {e}")
        continue
    vsurf = sorted(t for t in tables if "vsurf" in t.lower())
    print(f"[{lib}] {len(tables)} tables au total | vsurf*: {vsurf}")
# Resultat : tables annuelles vsurfd1996 -> vsurfd2025 (~30 ans), dans optionm et optionm_all.

[optionm_all] 402 tables au total | vsurf*: ['vsurfbr1996', 'vsurfbr1997', 'vsurfbr1998', 'vsurfbr1999', 'vsurfbr2000', 'vsurfbr2001', 'vsurfbr2002', 'vsurfbr2003', 'vsurfbr2004', 'vsurfbr2005', 'vsurfbr2006', 'vsurfbr2007', 'vsurfbr2008', 'vsurfbr2009', 'vsurfbr2010', 'vsurfbr2011', 'vsurfbr2012', 'vsurfbr2013', 'vsurfbr2014', 'vsurfbr2015', 'vsurfbr2016', 'vsurfbr2017', 'vsurfbr2018', 'vsurfbr2019', 'vsurfbr2020', 'vsurfbr2021', 'vsurfbr2022', 'vsurfbr2023', 'vsurfbr2024', 'vsurfbr2025', 'vsurfd1996', 'vsurfd1997', 'vsurfd1998', 'vsurfd1999', 'vsurfd2000', 'vsurfd2001', 'vsurfd2002', 'vsurfd2003', 'vsurfd2004', 'vsurfd2005', 'vsurfd2006', 'vsurfd2007', 'vsurfd2008', 'vsurfd2009', 'vsurfd2010', 'vsurfd2011', 'vsurfd2012', 'vsurfd2013', 'vsurfd2014', 'vsurfd2015', 'vsurfd2016', 'vsurfd2017', 'vsurfd2018', 'vsurfd2019', 'vsurfd2020', 'vsurfd2021', 'vsurfd2022', 'vsurfd2023', 'vsurfd2024', 'vsurfd2025']
[optionm] 578 tables au total | vsurf*: ['vsurfbr1996', 'vsurfbr1997', 'vsurfbr1998',

## 2) Colonnes reelles + maturites/deltas disponibles (annee temoin 2024)

In [ ]:
print('=== Colonnes de optionm.vsurfd2024 ===')
print(db.describe_table(library='optionm', table='vsurfd2024'))

print('\n=== Maturites (days) disponibles ===')
print(db.raw_sql('SELECT DISTINCT days FROM optionm.vsurfd2024 ORDER BY days')['days'].tolist())

print('\n=== Deltas disponibles ===')
print(db.raw_sql('SELECT DISTINCT delta FROM optionm.vsurfd2024 ORDER BY delta')['delta'].tolist())

print('\n=== cp_flag disponibles ===')
print(db.raw_sql('SELECT DISTINCT cp_flag FROM optionm.vsurfd2024')['cp_flag'].tolist())

# Resultat cle :
#  colonnes = secid, date, days, delta, impl_volatility, impl_strike, impl_premium, dispersion, cp_flag
#  days inclut 91 EXACTEMENT  -> notre maturite 3 mois est native
#  delta inclut +/-50         -> ATM dispo (call delta=50, put delta=-50)
#  ~536 M lignes / an -> FILTRER A LA SOURCE (days=91, delta in (50,-50), secid restreint)
#  NB : la colonne 'dispersion' ici = mesure de precision OptionMetrics, RIEN a voir avec notre signal.

=== Colonnes de optionm.vsurfd2024 ===
Approximately 535786868 rows in optionm.vsurfd2024.
              name  nullable              type  \
0            secid      True  DOUBLE PRECISION   
1             date      True              DATE   
2             days      True  DOUBLE PRECISION   
3            delta      True  DOUBLE PRECISION   
4  impl_volatility      True  DOUBLE PRECISION   
5      impl_strike      True  DOUBLE PRECISION   
6     impl_premium      True  DOUBLE PRECISION   
7       dispersion      True  DOUBLE PRECISION   
8          cp_flag      True        VARCHAR(1)   

                                             comment  
0                                        Security ID  
1                                               Date  
2                                 Days to Expiration  
3                                Delta of the Option  
4      Interpolated Implied Volatility of the Option  
5       The Strike Price Corresponding to this Delta  
6  The Premium of a The